In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="quora_cross_encoder_prefixed_sentence_tags_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "cross-encoder/quora-distilroberta-base"

model = CrossEncoder(model_name, device=str(device))
print(model_name)
print("num_labels:", model.model.config.num_labels)
print("id2label:", getattr(model.model.config, "id2label", None))


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/quora-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cross-encoder/quora-distilroberta-base
num_labels: 1
id2label: {0: 'LABEL_0'}


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sentence1 = ds["sentence1"]
sentence2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
prompted_pairs = [
    (f"Sentence 1: {s1}", f"Sentence 2: {s2}")
    for s1, s2 in zip(sentence1, sentence2)
]

scores = model.predict(
    prompted_pairs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

scores = np.asarray(scores)
print("scores_shape:", scores.shape)

if scores.ndim == 2 and scores.shape[1] == 2:
    positive_scores = scores[:, 1]
    y_pred = np.argmax(scores, axis=1).astype(int)
elif scores.ndim == 2 and scores.shape[1] == 1:
    positive_scores = scores[:, 0]
    y_pred = (positive_scores >= 0.5).astype(int)
elif scores.ndim == 1:
    positive_scores = scores
    y_pred = (positive_scores >= 0.5).astype(int)
else:
    raise ValueError(f"Unexpected prediction shape: {scores.shape}")

print("done")


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

scores_shape: (408,)
done


In [ ]:

vault.create_record_list("distilbert_cross_text_prediction", column_names=["prediction", "score"])

for i in range(len(y_pred)):
    vault.append_record("distilbert_cross_text_prediction", 
                        {
                            "prediction": y_pred[i],
                            "score": positive_scores[i] ,
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "distilbert_cross_text_prediction stores per-example model outputs generated by applying the cross-encoder/quora-distilroberta-base cross-encoder to the GLUE MRPC validation dataset, using prefixed inputs of the form \u201cSentence 1: \u2026\u201d and \u201cSentence 2: \u2026\u201d. Each record corresponds to one source sentence pair from glue_mrpc_validation and contains two fields: prediction, the predicted binary paraphrase label (0 = not paraphrase, 1 = paraphrase), and score, the model\u2019s positive-class score used to derive the prediction. In this workflow, this dataset serves as the prediction table for the validation run and is later joined with the original MRPC labels to compute accuracy, F1, error analysis, and the experiment summary."
embedding = get_embeddings(description)
vault.create_description("distilbert_cross_text_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "model predictions", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "cross-encoder/quora-distilroberta-base", "model_family": "cross-encoder", "input_format": "sentence-pair with prefixed tags", "input_prefixes": "Sentence 1:, Sentence 2:", "outputs": "prediction, score", "label_space": "0=not_paraphrase,1=paraphrase", "score_type": "positive class score", "domain": "news", "upstream_dataset": "glue_mrpc_validation"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_cross_text_prediction", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.40931372549019607, 'f1': 0.30547550432276654}
                precision    recall  f1-score   support

not_paraphrase       0.34      0.88      0.49       129
    paraphrase       0.78      0.19      0.31       279

      accuracy                           0.41       408
     macro avg       0.56      0.54      0.40       408
  weighted avg       0.64      0.41      0.36       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("prompted_sentence1:", prompted_pairs[i][0])
    print("prompted_sentence2:", prompted_pairs[i][1])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "score:", float(positive_scores[i]))


prompted_sentence1: Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
prompted_sentence2: Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 score: 0.08101906627416611
prompted_sentence1: Sentence 1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
prompted_sentence2: Sentence 2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 score: 0.002017710590735078
prompted_sentence1: Sentence 1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
prompted_sentence2: Sentence 2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 score: 0.0749620720744133
prompted_sentence1: S

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("prompted_sentence1:", prompted_pairs[i][0])
    print("prompted_sentence2:", prompted_pairs[i][1])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "score:", float(positive_scores[i]))

vault.create_record_list("quora_cross_encoder_prefixed_sentence_tags_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("quora_cross_encoder_prefixed_sentence_tags_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert_cross_text_prediction": [0, len(ds)]
                    })

summary

description = "This dataset stores the evaluation summary for the notebook run quora_cross_encoder_prefixed_sentence_tags_mrpc on the GLUE MRPC validation set. It contains a summary record of the cross-encoder paraphrase detection results produced from the prediction dataset distilbert_cross_text_prediction. Its fields are: accuracy (float, overall classification accuracy), f1 (float, binary F1 score for paraphrase detection), and classification_report (string, the full sklearn classification report with precision, recall, F1, and support for the not_paraphrase and paraphrase classes). In this workflow, it serves as the experiment-level metrics dataset that aggregates model performance over the full validation set and links the source inputs from glue_mrpc_validation and the generated predictions."
embedding = get_embeddings(description)
vault.create_description("quora_cross_encoder_prefixed_sentence_tags_mrpc_summary", description, embedding)

properties = {"dataset_type": "evaluation_summary", "task": "paraphrase_detection", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "cross-encoder/quora-distilroberta-base", "model_type": "cross_encoder", "input_format": "sentence_pair_with_prefixed_tags", "prompt_prefixes": "Sentence 1:/Sentence 2:", "metrics": ["accuracy", "f1", "classification_report"], "labels": ["not_paraphrase", "paraphrase"], "domain": "news", "output_granularity": "dataset_level_summary"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("quora_cross_encoder_prefixed_sentence_tags_mrpc_summary", cat, embedding, prop)



num_errors: 241
idx: 0
prompted_sentence1: Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
prompted_sentence2: Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0 score: 0.08101906627416611
idx: 3
prompted_sentence1: Sentence 1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
prompted_sentence2: Sentence 2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
true: 1 pred: 0 score: 0.017635010182857513
idx: 5
prompted_sentence1: Sentence 1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
prompted_sentence2: Sentence 2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
true: 1 pred: 0 score: 0.00964877288788557
idx: 7
prompted_sentence1: Sentence 1: This inte

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'cross-encoder/quora-distilroberta-base',
 'device': 'mps',
 'prompt_style': 'sentence_tags',
 'num_examples': 408,
 'accuracy': 0.40931372549019607,
 'f1': 0.30547550432276654}

In [ ]:
description = "quora_cross_encoder_prefixed_sentence_tags_mrpc is the TableVault experiment dataset for evaluating the cross-encoder/quora-distilroberta-base model on the GLUE MRPC validation split using prefixed inputs of the form \u201cSentence 1: \u2026\u201d and \u201cSentence 2: \u2026\u201d. The underlying source data comes from glue_mrpc_validation and contains one record per sentence pair with fields sentence1, sentence2, and label, where label indicates whether the pair is a paraphrase. This workflow produces a row-level prediction table, distilbert_cross_text_prediction, with fields prediction and score, linked back to the corresponding MRPC validation rows, and a summary table, quora_cross_encoder_prefixed_sentence_tags_mrpc_summary, with aggregate metrics accuracy, f1, and classification_report. Its role is to document both the model inputs and the derived evaluation outputs for a paraphrase detection experiment on MRPC." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("quora_cross_encoder_prefixed_sentence_tags_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "benchmark": "GLUE", "dataset": "MRPC", "source": "glue/mrpc", "split": "validation", "size": "408", "domain": "news", "model": "cross-encoder/quora-distilroberta-base", "model_type": "cross-encoder", "input_format": "prefixed sentence tags", "framework": "sentence-transformers", "outputs": "pairwise paraphrase predictions and evaluation summary"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("quora_cross_encoder_prefixed_sentence_tags_mrpc", cat, embedding, prop)